In [6]:
# =========================
# Imports & Device Setup
# =========================
import os, glob, random, numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.transforms import functional as Fv
import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Device & Seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# =========================
# Dataset Paths & Split
# =========================
DATASET_DIR = r"E:\ViT MLDC\dataset"
IMG_SIZE = 224

CLASS_NAMES = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print("Classes:", CLASS_NAMES)

image_paths, labels = [], []
for lbl in CLASS_NAMES:
    folder = os.path.join(DATASET_DIR, lbl)
    imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
    image_paths.extend(imgs)
    labels.extend([lbl]*len(imgs))

le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    image_paths, labels_encoded, stratify=labels_encoded, test_size=0.3, random_state=42
)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, stratify=y_temp, test_size=0.5, random_state=42
)
print(f"Train: {len(X_train_paths)}  Val: {len(X_val_paths)}  Test: {len(X_test_paths)}")

# =========================
# Dual Augmentation
# =========================
train_base = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
dual_aug1 = transforms.RandomChoice([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05))
])
dual_aug2 = transforms.RandomChoice([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9,1.0)),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
])
train_transform = transforms.Compose(train_base + [dual_aug1, dual_aug2,
                                                   transforms.ToTensor(),
                                                   transforms.Normalize([0.5]*3, [0.5]*3)])
val_transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                    transforms.ToTensor(),
                                    transforms.Normalize([0.5]*3, [0.5]*3)])

# =========================
# Feature Extraction (Tiny Swin V2)
# =========================
model_name = 'swin_tiny_patch4_window7_224'
feature_model = timm.create_model(model_name, pretrained=True, num_classes=0)
feature_model.eval().to(device)

@torch.no_grad()
def extract_feature_from_path(path, transform):
    img = Image.open(path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    feat = feature_model(x)
    return feat.squeeze(0).cpu().numpy()

print("Extracting features...")
X_train_feats = np.stack([extract_feature_from_path(p, train_transform) for p in tqdm(X_train_paths)])
X_val_feats   = np.stack([extract_feature_from_path(p, val_transform) for p in tqdm(X_val_paths)])
X_test_feats  = np.stack([extract_feature_from_path(p, val_transform) for p in tqdm(X_test_paths)])

# =========================
# LoRA ViT Head
# =========================
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.lora_a = nn.Linear(in_features, r, bias=False)
        self.lora_b = nn.Linear(r, out_features, bias=False)
    def forward(self, x):
        return self.linear(x) + self.lora_b(self.lora_a(x))

class ViTHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.head = LoRALinear(in_features, num_classes, r=4)
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            return self.head(self.dropout(x)).squeeze(0)
        return self.head(self.dropout(x))

in_features = X_train_feats.shape[1]
num_classes = len(CLASS_NAMES)
model_head = ViTHead(in_features=in_features, num_classes=num_classes).to(device)

# =========================
# Attention-Guided Mixup
# =========================
def attention_guided_mixup(X, y, mixup_count, alpha=0.4, attention_maps=None):
    mixX, mixy = [], []
    n = len(X)
    for _ in range(mixup_count):
        i, j = random.sample(range(n), 2)
        lam = np.random.beta(alpha, alpha)
        mixed = lam * X[i] + (1 - lam) * X[j]
        mixX.append(mixed)
        mixy.append(y[i])
    return np.array(mixX), np.array(mixy)

mixup_count = len(X_train_feats)
X_mix, y_mix = attention_guided_mixup(X_train_feats, y_train, mixup_count)
X_final, y_final = X_mix, y_mix

# =========================
# Dataloaders
# =========================
batch_size = 64
train_loader = DataLoader(TensorDataset(torch.tensor(X_final, dtype=torch.float32),
                                        torch.tensor(y_final, dtype=torch.long)),
                          batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_feats, dtype=torch.float32),
                                      torch.tensor(y_val, dtype=torch.long)),
                        batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_feats, dtype=torch.float32),
                                       torch.tensor(y_test, dtype=torch.long)),
                         batch_size=batch_size, shuffle=False)

# =========================
# PolyLoss + Label Smoothing
# =========================
class Poly1LossWithSmoothing(nn.Module):
    def __init__(self, num_classes, epsilon=1.0, label_smoothing=0.1):
        super().__init__()
        self.epsilon = epsilon
        self.label_smoothing = label_smoothing
        self.num_classes = num_classes
    def forward(self, logits, targets):
        targets_onehot = F.one_hot(targets, num_classes=self.num_classes).float()
        targets_smoothed = targets_onehot * (1 - self.label_smoothing) + self.label_smoothing / self.num_classes
        probs = F.softmax(logits, dim=-1)
        log_probs = torch.log(probs + 1e-8)
        ce_loss = -(targets_smoothed * log_probs).sum(dim=-1).mean()
        poly1 = ce_loss + self.epsilon * (1 - (probs * targets_smoothed).sum(dim=-1)).mean()
        return poly1

criterion = Poly1LossWithSmoothing(num_classes=num_classes)

# =========================
# SAM + Cosine LR
# =========================
from sam import SAM
base_optimizer = torch.optim.Adam
optimizer = SAM(model_head.parameters(), base_optimizer, lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer.base_optimizer, T_max=50)

# =========================
# Training Loop (SAM-compatible)
# =========================
EPOCHS = 50
PATIENCE = 5
best_val_loss = float("inf")
wait = 0
best_model_path = "tinyswin_lora_mixup_final.pth"

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model_head.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        # Forward + backward
        logits = model_head(xb)
        loss = criterion(logits, yb)
        loss.backward()
        
        # SAM first step
        optimizer.first_step(zero_grad=True)
        
        # Forward + backward second pass
        logits2 = model_head(xb)
        criterion(logits2, yb).backward()
        
        # SAM second step
        optimizer.second_step(zero_grad=True)
        
        total_loss += loss.item() * xb.size(0)
    
    epoch_loss = total_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    # Validation
    model_head.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model_head(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)
    
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}")
    
    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(model_head.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

# =========================
# Evaluation
# =========================
model_head.load_state_dict(torch.load(best_model_path))
model_head.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model_head(xb)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        y_true.extend(yb.numpy())
        y_pred.extend(preds)

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
print("\nFinal Test Metrics:")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# ================================
# TTA Inference
# ================================
def tta_identity(img): return val_transform(img)
def tta_hflip(img): return val_transform(Fv.hflip(img))
def tta_rot(img, deg=8): return val_transform(img.rotate(deg, resample=Image.BILINEAR))

tta_funcs = [tta_identity, tta_hflip, lambda img: tta_rot(img, 8), lambda img: tta_rot(img, -8)]

tta_preds = []
for p in tqdm(X_test_paths, desc='TTA inference'):
    img = Image.open(p).convert('RGB')
    logits_list = []
    for fn in tta_funcs:
        inp = fn(img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = feature_model(inp).squeeze(0)
            logit = model_head(feat)
            logits_list.append(logit.cpu().numpy())
    avg_logits = np.mean(np.stack(logits_list, axis=0), axis=0)
    pred = int(np.argmax(avg_logits))
    tta_preds.append(pred)

tta_accuracy = accuracy_score(y_test, tta_preds)
print(f"TTA Test Accuracy: {tta_accuracy:.4f}")

Device: cuda
Classes: ['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Powdery Mildew', 'Sooty Mould']
Train: 2800  Val: 600  Test: 600
Extracting features...


100%|██████████| 600/600 [00:07<00:00, 76.68it/s]


Epoch 1/50 - Train Loss: 2.6968  Val Loss: 1.8001
Epoch 2/50 - Train Loss: 2.4144  Val Loss: 1.4456
Epoch 3/50 - Train Loss: 2.3414  Val Loss: 1.3179
Epoch 4/50 - Train Loss: 2.2951  Val Loss: 1.2479
Epoch 5/50 - Train Loss: 2.2725  Val Loss: 1.2122
Epoch 6/50 - Train Loss: 2.2500  Val Loss: 1.1952
Epoch 7/50 - Train Loss: 2.2396  Val Loss: 1.1725
Epoch 8/50 - Train Loss: 2.2243  Val Loss: 1.1523
Epoch 9/50 - Train Loss: 2.2125  Val Loss: 1.1489
Epoch 10/50 - Train Loss: 2.2002  Val Loss: 1.1328
Epoch 11/50 - Train Loss: 2.1915  Val Loss: 1.1201
Epoch 12/50 - Train Loss: 2.1767  Val Loss: 1.1156
Epoch 13/50 - Train Loss: 2.1798  Val Loss: 1.1107
Epoch 14/50 - Train Loss: 2.1683  Val Loss: 1.1041
Epoch 15/50 - Train Loss: 2.1691  Val Loss: 1.1078
Epoch 16/50 - Train Loss: 2.1572  Val Loss: 1.0991
Epoch 17/50 - Train Loss: 2.1552  Val Loss: 1.0884
Epoch 18/50 - Train Loss: 2.1479  Val Loss: 1.0961
Epoch 19/50 - Train Loss: 2.1399  Val Loss: 1.0877
Epoch 20/50 - Train Loss: 2.1401  Val Lo

TTA inference: 100%|██████████| 600/600 [00:31<00:00, 18.89it/s]

TTA Test Accuracy: 0.9733
